# Lesson 2.5 — experiments: comparing strategies

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementations:

- `"mine"` uses your `task.py` files from lessons 2.2 (BFS, DFS), 2.3 (UCS) and 2.4 (greedy, A\*, `manhattan`); solve those tasks first;
- `"reference"` uses the course's reference implementations. Set `IMPLEMENTATION = "reference"` if one of your implementations does not work yet.

In [ ]:
import os
from pathlib import Path

from introai.testing.impl import load_module

# "mine" uses your task.py files from lessons 2.2-2.4; "reference" uses the course's versions.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    lessons = Path.cwd().parent
    lesson_2_2 = load_module(lessons / "02_bfs_dfs" / "task.py")
    lesson_2_3 = load_module(lessons / "03_uniform_cost_search" / "task.py")
    lesson_2_4 = load_module(lessons / "04_heuristics_and_a_star" / "task.py")
    bfs, dfs = lesson_2_2.bfs, lesson_2_2.dfs
    ucs = lesson_2_3.ucs
    greedy_best_first, a_star = lesson_2_4.greedy_best_first, lesson_2_4.a_star
    manhattan = lesson_2_4.manhattan
else:
    from introai.search.reference import a_star, bfs, dfs, greedy_best_first, ucs
    from introai.world.reference import manhattan

The function `compare(warehouse)` runs the five algorithms on one map and returns, for each, the number of steps of the route, its cost, the number of expansions and the time (the median of five runs). Greedy and A\* get the Manhattan distance to the dock. In the pictures blue cells are expanded, orange cells are the route, and green hatched cells are slow zones.

In [ ]:
import statistics
import time

import matplotlib.pyplot as plt

from introai.trace import TraceLimitExceeded, record
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"
LESSON = {"bfs": "2.2", "dfs": "2.2", "ucs": "2.3", "greedy_best_first": "2.4", "a_star": "2.4"}
HINT = 'or set IMPLEMENTATION = "reference" in the first cell'


def run(name, search, problem, h=None, limit=100_000):
    """Run one search; return the goal node and the expanded nodes."""
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = search(problem) if h is None else search(problem, h)
    except NotImplementedError:
        raise RuntimeError(
            f"{name} is not written yet - finish lesson {LESSON[name]} ({HINT})"
        ) from None
    except TraceLimitExceeded:
        raise RuntimeError(
            f"{name} expanded more than {limit} nodes - check lesson {LESSON[name]}"
        ) from None
    except TypeError as error:
        if "not supported between instances" not in str(error):
            raise
        raise RuntimeError(
            f"{name} compared two Node objects - add a counter on the heap"
        ) from None
    if node is None:
        raise RuntimeError(f"{name} found no route, but there is one - check lesson {LESSON[name]}")
    return node, [event.data["node"] for event in trace.of_kind("expand")]


def searches(warehouse):
    h = lambda cell: manhattan(cell, warehouse.dock)  # noqa: E731
    return [
        ("bfs", bfs, None),
        ("dfs", dfs, None),
        ("ucs", ucs, None),
        ("greedy_best_first", greedy_best_first, h),
        ("a_star", a_star, h),
    ]


def compare(warehouse):
    """Steps, cost, expansions and median time (ms) of the five searches on a map."""
    results = {}
    for name, search, h in searches(warehouse):
        node, expanded = run(name, search, RouteProblem(warehouse), h)
        times = []
        for _ in range(5):
            start = time.perf_counter()
            search(RouteProblem(warehouse)) if h is None else search(RouteProblem(warehouse), h)
            times.append((time.perf_counter() - start) * 1000)
        results[name] = (node, expanded, statistics.median(times))
    print(f"{'algorithm':>18} {'steps':>6} {'cost':>5} {'expanded':>9} {'ms':>6}")
    for name, (node, expanded, ms) in results.items():
        print(
            f"{name:>18} {len(node.solution()):>6} {node.path_cost:>5g} "
            f"{len(expanded):>9} {ms:>6.2f}"
        )
    return results

## Experiment 1: five algorithms on maps with costs

You know the map `congested.txt` from lessons 2.3 and 2.4. The map `depot.txt` is a larger warehouse with two crowded cross aisles.

In [ ]:
congested = WarehouseMap.from_file(MAPS / "congested.txt")
depot = WarehouseMap.from_file(MAPS / "depot.txt")
print(render_text(depot))

**Your prediction.** On the map `depot.txt`: which algorithms will find the cheapest route? Which will expand the fewest cells? Which will find the longest route?

*My prediction:* …

In [ ]:
print("congested.txt")
compare(congested)
print()
print("depot.txt")
depot_results = compare(depot)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 17))
for ax, (name, (node, expanded, _)) in zip(axes, depot_results.items()):
    title = f"{name}: {len(node.solution())} steps, cost {node.path_cost:g}"
    plot_map(depot, path=node.states(), expanded=[n.state for n in expanded], title=title, ax=ax)
fig.tight_layout()

**Think about it.** Compare the results with the table of properties from Part A. Did any algorithm behave "better" on these maps than the table would suggest, for example DFS finding a fairly cheap route? Why does that not change the answers in the table? How many cells does A\* save over UCS on each map, and where does the difference come from?

## Experiment 2: memory

We measure memory by the largest size of the frontier. Your code does not report it, so in this experiment we always use the course's reference versions, which report the frontier size after every step.

**Your prediction.** Which algorithm has the larger frontier, BFS or DFS? Is it the same on every map?

*My prediction:* …

In [ ]:
from introai.search import reference
from introai.testing.search import max_frontier

names = ["aisles", "open_floor", "congested", "depot"]
largest = {"BFS": [], "DFS": []}
for name in names:
    warehouse = WarehouseMap.from_file(MAPS / f"{name}.txt")
    for label, search in [("BFS", reference.bfs), ("DFS", reference.dfs)]:
        with record(kinds={"frontier"}) as trace:
            search(RouteProblem(warehouse))
        largest[label].append(max_frontier(trace))
    print(f"{name:>11}: largest frontier BFS {largest['BFS'][-1]:>3}, DFS {largest['DFS'][-1]:>3}")

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.38
for offset, (label, color) in zip(
    (-width / 2, width / 2), [("BFS", "#2a78d6"), ("DFS", "#eb6834")]
):
    positions = [i + offset for i in range(len(names))]
    ax.bar(positions, largest[label], width=width - 0.04, color=color, label=label)
ax.set_xticks(range(len(names)), names)
ax.set_title("Largest frontier during the search", loc="left", color="#0b0b0b")
ax.set_ylabel("nodes in the frontier", color="#52514e")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False, labelcolor="#0b0b0b")
fig.tight_layout()

**Think about it.** On which map does DFS have its largest frontier, and why exactly there? Look at the length of the DFS route on that map. What does the formula O(*b*·*m*) say when *m* is large?

## Experiment 3: where exact search ends

We go back to the parcel pickup problem of lesson 1.3 (the course's reference version). A\* looks for the shortest plan to collect 1, 2, …, 6 parcels on the map with aisles, once with *h* = 0 and once with the heuristic "the number of parcels still lying on the floor".

**Your prediction.** How many times does the number of expansions grow with each extra parcel? Does the simple heuristic change anything?

*My prediction:* …

In [ ]:
from introai.world.reference import ParcelPickupProblem

aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
CELLS = [(3, 1), (5, 4), (9, 21), (13, 15), (1, 20), (7, 13)]
counts = list(range(1, len(CELLS) + 1))
work = {"h = 0": [], "parcels left": []}
for k in counts:
    problem = ParcelPickupProblem(aisles.with_parcels(CELLS[:k]))
    for label, h in [("h = 0", lambda state: 0), ("parcels left", lambda state: len(state[1]))]:
        node, expanded = run("a_star", a_star, problem, h, limit=500_000)
        work[label].append(len(expanded))
    print(
        f"{k} parcels: plan {len(node.solution())} actions, expanded "
        f"{work['h = 0'][-1]} (h = 0) and {work['parcels left'][-1]} (parcels left)"
    )

fig, ax = plt.subplots(figsize=(7, 4))
for label, color in [("h = 0", "#2a78d6"), ("parcels left", "#eb6834")]:
    ax.plot(
        counts,
        work[label],
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=label,
    )
ax.set_yscale("log")
ax.set_title("Nodes expanded by A* to collect the parcels", loc="left", color="#0b0b0b")
ax.set_xlabel("number of parcels", color="#52514e")
ax.set_ylabel("nodes expanded (log scale)", color="#52514e")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False, labelcolor="#0b0b0b")
fig.tight_layout()

**Think about it.** The chart uses a logarithmic scale. How many expansions would A\* need for 20 parcels if the growth keeps its pace? Why does the heuristic "number of parcels" help hardly at all? You will design better heuristics in lesson 2.7, and meet methods for really large problems in module 4.

## Summary

Go back to the review questions in the lesson. After your own attempt, you can compare your results and conclusions with the answers in `solutions/02_search/05_comparing_strategies/answers.md`.